# Verifica se un albero binario è di ricerca

In [2]:
from typing import override

from alberi.alberibinari import AlberoBin

class abr(AlberoBin):
    def diRicerca(self):
        return abr._diRicercaRic(self, float('-inf'), float('inf'))
    @staticmethod
    def _diRicercaRic(nodo, min:int, max:int):
        if nodo is None:
            return True
        if nodo.val <= min or nodo.val >= max:
            return False
        return abr._diRicercaRic(nodo.sin, min, nodo.val) and abr._diRicercaRic(nodo.des.val, max)

# Dato un albero binario verificare se è bilanciato
con un algoritmo ricorsivo di complessità lineare

In [3]:
class abb(AlberoBin):
    def is_bilanciato(self):
        return abb._is_bilanciato(self)[1]
    @staticmethod
    def _is_bilanciato(nodo:AlberoBin) -> tuple[int, bool]:
        if nodo is None:
            return (0, True)
        resSin = abb._is_bilanciato(nodo.sin)
        resDes = abb._is_bilanciato(nodo.des)
        if (not resDes[1] or not resSin[1]):
            return (0, False)
        return (max(resSin[0], resDes[0])+1, abs(resSin[0] - resDes[0]) <= 1)

    # versione incredibilmente pulita
    def is_bilanciato2(self):
        try:
            abb._is_bilanciato2(self)
            return True
        except ValueError:
            return False
    @staticmethod
    def _is_bilanciato2(nodo:AlberoBin) -> int:
        if nodo is None:
            return 0
        hSin = abb._is_bilanciato2(nodo.sin)
        hDes = abb._is_bilanciato2(nodo.des)
        if abs(hSin-hDes) > 1:
            raise ValueError("non bilanciato")
        return max(hSin, hDes) + 1

# Visita sui grafi

In [4]:
from grafi.grafi import GrafoMA
from grafi.grafi import GrafoLA
from grafi.grafino import GrafoLANO
from grafi.grafino import GrafoMANO

class Visita(GrafoLA):
    def _visita_prof(self, nodo: int, visitati: list[bool], res: list[int]):
        if not visitati[nodo]:
            visitati[nodo] = True
            res.append(nodo)
            for a in self.adiacenti(nodo):
                self._visita_prof(a, visitati, res)
    def visita_prof(self, nodo: int) -> list[int]:
        res: list[int] = []
        visitati: list[bool] = [False for _ in range(self.n)]
        self._visita_prof(nodo, visitati, res)
        return res

    def visita_vent(self, nodo:int) -> list[int]:
        res: list[int] = [nodo]
        visitati: list[bool] = [False for _ in range(self.n)]
        coda: list[int] = [nodo]
        visitati[nodo] = True
        while(coda):
            curr: int = coda.pop(0)
            for a in self.adiacenti(curr):
                if not visitati[a]:
                    visitati[a] = True
                    coda.append(a)
                    res.append(a)
        return res

# Verifica se un grafo è un albero / verifica aciclicità

In [5]:
from grafi.grafi import GrafoMA
from grafi.grafi import GrafoLA
from grafi.grafino import GrafoLANO
from grafi.grafino import GrafoMANO

class GrafoAlberoOR(Visita):
    def verificaAlberoOR(self):
        return self.n == self.m + 1 and self.econnessoOR(0)
    def econnessoOR(self, nodo: int) -> bool:
        visita = self.visita_vent(nodo)
        if len(visita) == self.n:
            return True
        return False
    def eaciclicoOR(self):
        visitati: list[bool] = [False for _ in range(self.n)]
        k: int = 0
        for i in range(self.n):
            if not visitati[i]:
                k += 1
                self._visita_prof(i, visitati, [])
        return self.n < self.m + k

class GrafoAlberoNO(GrafoLANO):
    def aciclicoNO(self):
        rimossi: list[bool] = [False for _ in range(self.n)]
        gradi: list[int] = [0 for _ in range(self.n)]
        for a in self.archi():
            gradi[a[1]] += 1
        curr = self.trovaZero(gradi, rimossi)
        while curr != -1:
            rimossi[curr] = True
            for a in self.adiacenti(curr):
                gradi[a] -= 1
            curr = self.trovaZero(gradi, rimossi)
        return self.tuttiZeri(gradi)
    def tuttiZeri(self, gradi: list[int]):
        for i in gradi:
            if i != 0:
                return False
        return True
    def trovaZero(self, gradi: list[int], rimossi: list[bool]):
        for i in range(len(gradi)):
            if gradi[i] == 0 and not rimossi[i]:
                return i
        return -1

# Scheduling delle attività
Caso pratico di utilizzo di un grafo aciclico orientato (DAG).

- Ogni nodo del grafo rappresenta un'attività da svolgere.
- Ogni arco rappresenta una relazione di propedeuticità tra due attività (es. A→B significa "A deve terminare prima che B possa iniziare").
- Ogni attività ha un proprio tempo di esecuzione.
- Hai a disposizione un numero illimitato di esecutori (quindi più attività possono partire in parallelo, appena le loro propedeuticità sono soddisfatte).
- Si parte al tempo 0.

Domanda: qual è il tempo minimo necessario per completare tutte le attività, rispettando i vincoli di propedeuticità?

In [6]:
class Scheduler(GrafoAlberoOR):
    def scheduling(self) -> int:
        t: int = 0
        visitati: list[bool] = [False for _ in range(self.n)]
        gradi: list[int] = [0 for _ in range(self.n)]
        for p, a in self.archi():
            gradi[a] += 1
        esec: list[tuple[int, int]] = []
        esec.extend(self.trovazeri(gradi, visitati))
        while esec:
            curr = min(esec, key = lambda x: x[1])
            esec.remove(curr)
            t += curr[1]
            for i in range(len(esec)):
                esec[i] = (esec[i][0], esec[i][1] - curr[1])
            for a in self.adiacenti(curr[0]):
                gradi[a] -= 1
            esec.extend(self.trovazeri(gradi, visitati))
        return t

    # trova tutti quelli che hanno grado di ingresso 0, li aggiunge a una lista associandoci il tempo rimanente alla fine dell'esecuzione e li restituisce
    def trovazeri(self, gradi: list[int], visitati: list[bool]) -> list[tuple[int, int]]:
        res: list[tuple[int, int]] = []
        for i in range(len(gradi)):
            if gradi[i] == 0 and not visitati[i]:
                visitati[i] = True
                res.append((i,i))
        return res

In [7]:
from grafi.grafi import GrafoLA
from grafi.grafi import GrafoMA

# Scheduler, Visita e GrafoAlberoOR li presumo già definiti in celle precedenti del notebook


def costruisci_dag() -> Scheduler:
    # 5 nodi, indice == durata (nodo 0 dura 0, nodo 3 dura 3, ecc.)
    g = Scheduler(5)
    g.aggiungiarco(0, 1)
    g.aggiungiarco(0, 2)
    g.aggiungiarco(1, 3)
    g.aggiungiarco(2, 3)
    g.aggiungiarco(3, 4)
    return g


def main():
    g = costruisci_dag()

    # Cammino critico atteso: 0 -> 2 -> 3 -> 4 = durate 0+2+3+4 = 9
    # (l'altro ramo 0 -> 1 -> 3 -> 4 = 0+1+3+4 = 8, quindi non è quello critico)
    atteso = 9
    risultato = g.scheduling()

    print(f"Tempo minimo calcolato: {risultato}")
    print(f"Atteso: {atteso}")
    assert risultato == atteso, f"FALLITO: atteso {atteso}, ottenuto {risultato}"
    print("Test superato.")


if __name__ == "__main__":
    main()

Tempo minimo calcolato: 9
Atteso: 9
Test superato.


# Esercizio
Dato un grafo non orientato pesato (pesi = costo di percorrenza), calcolare la centralità di ogni nodo, definita come il numero di cammini minimi (tra tutte le coppie di nodi) che passano per quel nodo come nodo intermedio.

L'idea è identificare i nodi "cruciali" per la connettività del grafo: rimuovendo un nodo ad alta centralità, il grafo diventa più difficile da attraversare (applicazioni: interdizione militare di uno snodo, chiusura di un nodo per rallentare un contagio).

Richiesta: modificare l'algoritmo di Floyd-Warshall in modo da, oltre alle distanze minime, tenere traccia anche del numero di cammini minimi che passano per ciascun nodo intermedio, tenendo conto che possono essercene più di uno (potenzialmente esponenziali) per coppia.

# Backtracking
## N regine
Date N regine e una scacchiera N×N, posiziona tutte le N regine sulla scacchiera in modo che nessuna coppia di regine si attacchi a vicenda.
Vincoli: due regine si attaccano se si trovano sulla stessa riga, stessa colonna, o stessa diagonale (principale o secondaria).
Task tipico da esercizio:

1. Trovare una soluzione (una qualsiasi disposizione valida)
2. Trovare tutte le soluzioni
3. Contare il numero totale di soluzioni

Traccia consigliata (stile backtracking):

Rappresenta la soluzione come un array pos[] di lunghezza N, dove pos[i] = colonna in cui si trova la regina della riga i (questo elimina automaticamente il conflitto "stessa riga")
Costruisci la soluzione riga per riga: alla riga i, prova ogni colonna c da 0 a N-1
Prima di piazzare, verifica compatibilità con le regine già piazzate nelle righe precedenti (colonna diversa, diagonali diverse: |pos[i] - pos[j]| != |i - j|)
Se compatibile, piazza e ricorsione sulla riga successiva; altrimenti backtrack

In [2]:
from backtracking.backtracking import ProblemaBack

class Nregine(ProblemaBack):
    N = 9
    pos: list[int] = [-1 for _ in range(N)] # i indice di riga, pos[i] indice di colonna


    def primaScelta(self, liv: int):
        self.pos[liv] = 0
        return True

    def successivaScelta(self, liv: int)->bool:
        if self.pos[liv] >= self.N-1:
            return False
        self.pos[liv] += 1
        return True

    def solCompleta(self, liv: int)->bool:
        for i in range(self.N):
            if self.pos[i] == -1:
                return False
        return True

    def verificaVincoli(self, liv: int)->bool:
        for i in range(liv):
            if self.pos[i] == self.pos[liv] or abs(self.pos[i] - self.pos[liv]) == abs(i-liv):
                return False
        return True

    def costruisciSoluzione(self, liv: int):
        sol: list[list[int]] = [[0 for _ in range(self.N)] for _ in range(self.N)]
        for i in self.pos:
            sol[i][self.pos[i]] = 1
        for i in range(self.N):
            for j in range(self.N):
                print(sol[i][j], end="")
            print()

def main():
    r = Nregine().risolvi()

if __name__ == "__main__":
    main()

100000000
001000000
000001000
000000010
010000000
000100000
000000001
000000100
000010000


# Pseudocodici algoritmi noti
## Prim

In [ ]:
res = nodo1
taglio: heap = {G.n}
taglio.aggiungiT(nodo1) # aggiunge al taglio una coppia nodo-peso per tutti gli adiacenti di nodo
while res.n < G.n: # alb.n numero di nodi dell'albero e G.n numero di nodi del grafo
    nuovo = estrai_min(taglio) # estrae la coppia nodo-peso con peso minore e la rimuove
    res.add(nuovo) # aggiunge all'albero il nuovo nodo e il relativo arco
    taglio.aggiungiT(nuovo[1]) # aggiunge al taglio la coppia nodo-peso di tutti gli adiacenti, se è già presente il nodo verifica se il nuovo arco ha un costo minore
return res

## Esercizio 1
Si consideri una classe `AlberoB` che rappresenta alberi binari in cui la parte informativa di ogni nodo è un numero intero. Si assuma che in tale classe siano implementati i seguenti metodi:
```
public interface AlberoB {
/* restituisce il sottoalbero destro dell’albero corrente, la complessità temporale è (1)*/
public AlberoB destro( );
/* restituisce il sottoalbero sinistro dell’albero corrente, la complessità temporale è (1)*/
public AlberoB sinistro( );
/* restituisce il valore memorizzato nella radice dell’albero, la complessità temporale è (1)*/
public int val( );
}
```
Si deve realizzare un metodo `public static boolean eRipetuto (AlberoB a, int x) {…}` che restituisce `true` se e solo se vi è almeno un nodo `n` nell’albero a tale che l’intero `x` appare sia nel sottoalbero sinistro che nel sottoalbero destro di `n`.

In [8]:
from alberi.alberibinari import AlberoBin

class es1(AlberoBin):
    def eRipetuto(self, nodo: AlberoBin, x: int) -> bool:
        if nodo is None:
            return False
        return (es1.ricerca(nodo.des, x) and es1.ricerca(nodo.sin, x)) or self.eRipetuto(nodo.des, x) or self.eRipetuto(nodo.sin, x)
    @staticmethod
    def ricerca(nodo: AlberoBin, x: int):
        if nodo is None:
            return False
        return nodo.val == x or es1.ricerca(nodo.des, x) or es1.ricerca(nodo.sin, x)

### Complessità
**Caso migliore**

Nel caso migliore `x` è figlio della radice dell'albero, e la complessità sarà quindi costante, sia spaziale che temporale $T_m, S_m \in \Theta(1)$

**Caso peggiore**

Nel caso peggiore l'albero è degenere, l'algoritmo scorre tutti i nodi in `eRipetuto()` e per ogni nodo lancia una chiamata a `ricerca()` che a sua volta effettua una visita, comportando una complessità temporale $T_p \in \Theta(n^2)$. La spaziale invece è lineare in quanto il numero di chiamate contemporaneamente sullo stack è pari al numero di nodi $S_p \in \Theta(n)$

## Esercizio 2
verificare `almenoUnDoppio(a: AlberoBin, x: int)` che restituisce `True` se e solo se esiste `x` che appare in un nodo non foglia e appare in un nodo foglia di `a`

In [9]:
def almenoUnDoppio(a: AlberoBin, x: int) -> bool:
    return ricFoglia(a, x) and ricMid(a, x)
def ricFoglia(a, x: int) -> bool:
    if a is None:
        return False
    if a.des is None and a.sin is None:
        return a.val == x
    return ricFoglia(a.sin, x) or ricFoglia(a.des, x)
def ricMid(a, x: int) -> bool:
    if a is None or (a.des is None and a.sin is None):
        return False
    return a.val == x or ricMid(a.sin, x) or ricMid(a.des, x)

### Complessità
**Caso migliore**

Nel caso migliore il figlio sinistro della radice è una foglia e contiene `x`, come la radice. Le chiamate ricorsive non dipendono dalla grandezza dell'input in quanto non facciamo ipotesi sulla forma del sottoalbero destro: $T_m(n) = S_m(n) \in \Theta(1)$

**Caso peggiore**

Nel caso peggiore l'albero è degenere e la sua esplorazione comporterà una complessità $T_p(n) = S_p(n) \in \Theta(n)$

## Esercizio 3
`foglieM(a: AlberoBin)` restituisce `True` se e solo se esiste solo una foglia dell'albero che ha valore $\geq 0$

In [10]:
def foglieM(a:AlberoBin) -> bool:
    return conta(a) == 1
def conta(a) -> int:
    if a is None:
        return 0
    if a.des is None and a.sin is None:
        if a.val >= 0:
            return 1
        return 0
    return conta(a.sin) + conta(a.des)

### Complessità
Per produrre l'output l'algoritmo legge tutte le foglie quindi la complessità temporale, indipendentemente dal caso, è $T_m(n) = T_p(n) \in \Theta(n)$.

La complessità spaziale invece sarà lineare nel caso peggiore se l'albero è degenere e logaritmica nel caso migliore se l'albero è bilanciato $S_m(n) \in \Theta(log_2(n))$ e $S_p(n) \in \Theta(n)$

## Esercizio 4
Restituisci il livello del nodo con valore `x`, se non è presente restituisce `0`

In [11]:
def livelloX(a, x: int) -> int:
    return calcolo(a, x, 0)
def calcolo(a, x: int, curr: int) -> int:
    if a is None:
        return 0
    if a.val == x:
        return curr
    return max(calcolo(a.sin, x, curr + 1), calcolo(a.des, x, curr + 1))

calcola l'altezza del nodo con valore `x`, se non è presente restituisce `0`

In [12]:
def altezzaX(a, x: int) -> int:
    if a is None:
        return 0
    if a.val == x:
        return calcoloH(a, x, 0)
    return max(altezzaX(a.sin, x), altezzaX(a.des, x))
def calcoloH(a, x: int, curr: int) -> int:
    if a is None:
        return curr
    return max(calcoloH(a.sin, x, curr + 1), calcoloH(a.des, x, curr + 1))

### Complessità
**Temporale**

In entrambi gli algoritmi a prescindere dall'output verrà esplorato l'albero nella sua interezza, di conseguenza $T(n) \in \Theta(n)$.

**Spaziale**

La spaziale invece dipende dal numero di chiamate contemporaneamente sullo stack, che nel caso migliore, in un albero bilanciato sarà dato da $S_m(n) \in \Theta(log_2(n))$ , mentre nel caso peggiore, in un albero degenere tutte le chiamate sono sullo stack $S_p(n) \in \Theta(n)$

## Esercizio 5
Si definisca un metodo che restituisce true se esiste un nodo non foglia di valore v per cui la somma dei valori contenuti nel suo sottoalbero sinistro e destro sia minore dello stesso

In [ ]:
def verifica(a: AlberoBin) -> bool:
    if a is None or isFoglia(a):
        return False
    if a.val < (somma(a)-a.val):
        return True
    return verifica(a.sin) or verifica(a.des)
def somma(a: AlberoBin) -> bool:
    if a is None:
        return 0
    return a.val + somma(a.sin) + somma(a.sin)

## Esercizio 6
Si definisca un metodo che restituisce true se e solo se invertendo sottoalbero sinistro e destro l'albero non cambia

In [ ]:
def verifica(a: AlberoBin) -> bool:
    if a is None:
        return False
    return uguale(a.sin, a.des)
def uguale(a1: AlberoBin, a2:AlberoBin) -> bool:
    if a1 is None and a2 is None:
        return True
    if a1 is None or a2 is None:
        return False
    return a1.val == a2.val and uguale(a1.sin, a2.sin) and uguale(a1.des, a2.des)

## Esercizio 8
Si realizzi un metodo `verificaNodoLivello(AlberoBinario a, int l)` che restituisce true se esiste un nodo n che appare in a ad un livello $l'>= l$ avente valore maggiore di $l'$.

In [ ]:
def verificaNodoLivello(a: AlberoBin, l: int) -> bool:
    if a is None:
        return False
    return verificaRic(a, l, 0)
def verificaRic(a: AlberoBin, l: int, curr: int) -> bool:
    if a is None:
        return False
    if curr < l and a.val <= curr:
        return verificaRic(a.sin, l, curr + 1) or verificaRic(a.des, l, curr + 1)
    return True

## Esercizio 9
Restituisce true se e solo se esiste almeno un valore che appare in un nodo n di a che si trova ad un livello liv e tale valore appare anche in b in un nodo n' che si trova ad un livello l' maggiore di liv

In [ ]:
def cercaNodo(b: AlberoBin, n: int, l: int, curr: int) -> bool:
    if b is None or curr > l:
        return False
    if b.val == n:
        return True
    return cercaNodo(b.sin, l, n, curr + 1) or cercaNodo(b.des, l, n, curr +1)
def verifica(a: AlberoBin, b: AlberoBin, l: int) -> bool:
    if a is None:
        return False
    if cercaNodo(b, a.val, l, 0):
        return True
    return verifica(a.sin, b, l+1) or verifica(a.des, b, l+1)

## Esercizio 10
Restituisce true se e solo se esiste almeno un valore che appare in un nodo n di a che si trova ad un livello liv e tale valore appare anche in a in un nodo n' che si trova ad un livello l' maggiore di liv ma che non è un discendente di n.

In [ ]:
def esisteN(a: AlberoBin) -> bool:
    if a is None or eFoglia(a):
        return False
    return cercaVal(a.sin, a.des) or cercaVal(a.des, a.sin) or esisteN(a.sin) or esisteN(a.des)
def cercaVal(a: AlberoBin, nodo: AlberoBin) -> bool:
    if nodo is None or a is None:
        return False
    return a.val == nodo.val or cercaVal(a.sin, nodo) or cercaVal(a.des, nodo)

## Esercizio 11
restituisce true se e solo se tutti i nodi di a contengono un valore maggiore del valore contenuto nella radice del loro sottoalbero sinistro e minore del valore contenuto nella radice del loro sottoalbero destro nel caso che tali sottoalberi non siano null

In [ ]:
def verificaIntervallo(a:AlberoBin) -> bool:
    if a is None or eFoglia(a):
        return True
    if a.des is None:
        return a.val > a.sin.val and verificaIntervallo(a.sin)
    if a.sin is None:
        return a.val < a.des.val and verificaIntervallo(a.des)
    return a.val > a.sin.val and a.val < a.des.val and verificaIntervallo(a.sin) and verificaIntervallo(a.des)

## Esercizio 12
Restituisce true se esiste almeno un nodo foglia n in a il cui valore é non maggiore (<=) di tutti i nodi antenati

In [ ]:
def fogliolina(a: AlberoBin) -> bool:
    return fogliolinaRic(a, float('inf'))
def fogliolinaRic(a: AlberoBin, minPrec: int) -> bool:
    if a is None:
        return False
    if eFoglia(a):
        return a.val <= minPrec
    new = min(a.val, minPrec)
    return fogliolinaRic(a.sin, new) or fogliolinaRic(a.des, new)

## Esercizio 13
metodo che restituisce true sse un valore appare almeno due volte nell'albero in due nodi diversi

In [ ]:
def almenoDue(a:AlberoBin) -> bool:
    if a is None:
        return False
    return contaRic(a, a.val) >= 2 or almenoDue(a.sin) or almenoDue(a.des)
def contaRic(a: AlberoBin, val: int) -> int:
    if a is None:
        return 0
    if a.val == val:
        return 1 + contaRic(a, val)
    return contaRic(a.sin, val)

## Esercizio 15
Restituisce true se e solo se vi é almeno un nodo n tale che l'intero x appartiene sia al suo sottoalbero destro che al sinistro

In [ ]:
def appartiene(a: AlberoBin, x: int) -> bool:
    if a is None:
        return False
    return (compare(a.sin, x) and compare(a.des, x)) or appartiene(a.sin, x) or appartiene(a.des, x)
def compare(a:AlberoBin, x: int) -> bool:
    if a is None:
        return False
    return a.val == x or compare(a.sin, x) or compare(a.des, x)

## Esercizio 16
Restituisce true sse gli alberi a e b non sono uno la copia dell'altro

In [ ]:
def nonUguali(a: AlberoBin, b: AlberoBin) -> bool:
    if a is None and b is None:
        return False
    if a is None or b is None:
        return True
    return a.val != b.val or nonUguali(a.sin, b.sin) or nonUguali(a.des, b.des)

## Esercizio 17
True sse esistono almeno due nodi dell'albero che soddisfano la seguente proprietá: la somma tra il valore contenuto nel nodo ed il valore del livello in cui si trova é minore di zero

In [ ]:
def verificaSomma(a: AlberoBin) -> bool:
    try:
        verificaRic(a, 0)
    except ValueError:
        return True
    return False
def verificaRic(a: AlberoBin, liv: int) -> int:
    if a is None:
        return 0
    count = verificaRic(a.sin, liv + 1) + verificaRic(a.des, liv + 1)
    if a.val + liv < 0:
        count += 1
    if count >= 2:
        raise ValueError
    return count

## Esercizio 18
True sse per ogni foglia di a vale la condizione che il padre di tale foglia contiene un valore maggiore di zero

In [ ]:
def condizione(a: AlberoBin):
    if a is None:
        return True
    if (eFoglia(a.sin) or eFoglia(a.des)) and a.val <= 0:
        return False
    return condizione(a.sin) and condizione(a.des)
def eFoglia(nodo):
    if nodo is None:
        return False
    if nodo.des is None and nodo.sin is None:
        return True
    return False

## Esercizio 19
restituisce true sse esiste un nodo che si trova al livello l, che ha valore 0 e i cui discendenti hanno tutti valore 0

In [ ]:
def tutti0(a: AlberoBin, liv: int) -> bool:
    if a is None:
        return False
    return tutti0ric(a, liv, 0)
def tutti0ric(a: AlberoBin, liv: int, curr: int) -> bool:
    if a is None:
        return False
    if liv == curr:
        return verifica0(a)
    return tutti0ric(a.sin, liv, curr + 1) or tutti0ric(a.des, liv, curr + 1)
def verifica0(a: AlberoBin) -> bool:
    if a is None:
        return True
    return a.val == 0 and verifica0(a.sin) and verifica0(a.des)

## Esercizio 20
restituisce il numero di valori pari memorizzati nei nodi di a al livello k ( si assuma che k sia un numero
non negativo, che la radice abbia livello 0, e che il metodo debba restituire 0 nel caso in cui a non
contenga nodi al livello k

In [ ]:
def pariK(a: ALberoBin, k: int) -> int:
    if a is None:
        return 0
    return pariKric(a, k, 0)
def pariKric(a: ALberoBin, k: int, curr: int) -> int:
    if a is None:
        return 0
    if curr == k:
        return 1 if a.val % 2 == 0 else 0
    return pariKric(a.sin, k, curr + 1) + pariKric(a.des, k, curr + 1)

## Esercizio 21
restituisce true sse esistono due nodi diversi n ed m al livello l tali che `n.val() = m.val()`

In [ ]:
def nodiUguali(a: AlberoBin, l: int) -> bool:
    if a is None:
        return False
    livello: list[int] = listaLiv(a, l, 0)
    for i in range(len(livello)):
        for j in range(i+1, len(livello)):
            if livello[i] == livello[j]:
                return True
    return False
def listaLiv(a: AlberoBin, l: int, curr: int) -> list[int]:
    if a is None:
        return []
    if curr == l:
        return [a.val]
    return listaLiv(a.sin, l, curr + 1) + (listaLiv(a.des, l, curr + 1))